<a href="https://colab.research.google.com/github/Zeshan811/StarterNotebookA1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane (Lane 3 — Structured Content Archetype Clustering) is a clustering task.

It is not classification, because I have no pre-existing label to predict (like "declining" or "not declining") — I don't know the groups in advance. It is not ranking, because I'm not ordering pages by priority. It is not scoring, because I'm not assigning a single opportunity number per page. Instead, I want the algorithm to look at several numeric signals per page (impressions, engagement, freshness, CTR, etc.) and discover natural groupings on its own — pages that behave similarly get placed together. This is unsupervised learning: there is no "right answer" column in the data to check against, only patterns I inspect afterward.

One-paragraph frame:
For a content strategist deciding which pages to review first, we will build cluster group profiles from the starter content dataset, grouping pages by impressions, engagement rate, CTR, average position, freshness, and word count — measured by silhouette score plus manual inspection of cluster averages. A wrong call costs wasted review time on a low-value page, or a missed opportunity on a page that needed attention. A plain rule isn't enough because archetypes depend on the combination of six signals at once, not any single threshold. We will claim only observed and decision-support results — not causal or semantic claims.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There is no target column in the traditional sense, because clustering is unsupervised — there is nothing to "predict." Instead of a target, what matters is the feature set I feed the algorithm, since the clusters are entirely defined by those features.

Planned features (all observed signals, no product decision flags):
- impressions_90d (demand/visibility)
- engagement_rate (how well the page holds attention)
- ctr (click efficiency at its position)
- avg_position (search ranking position)
- days_since_last_update (freshness)
- word_count (content depth)

After clustering runs, the cluster label (0, 1, 2, ...) becomes a kind of proxy output — but it is a group membership I define and name myself by inspecting each cluster's averages, not an observed real-world outcome like "this page later declined." I will be careful not to call the cluster number a "ground truth" label — it's a lens I create, not a fact I discover.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship-starter


## 3. Success metric

*One metric you can defend. What number means 'good'?*

For clustering, standard accuracy/precision metrics don't apply, since there is no ground-truth label. Instead I will use the silhouette score as my primary defensible metric.

The silhouette score measures, for every page, how similar it is to pages in its own cluster compared to pages in the nearest other cluster. It ranges from -1 to 1:
- close to 1 = the page fits its cluster well and is far from other clusters (good separation)
- close to 0 = the page sits on the border between two clusters
- negative = the page is probably in the wrong cluster

'Good' for my project means an average silhouette score meaningfully above 0 (roughly 0.25+ is a reasonable working target for messy real-world data like this), combined with a manual sanity check: when I look at 5-10 real pages from each cluster, do their numbers actually match the story I'm telling about that cluster? A high silhouette score with clusters that don't make sense on inspection would not count as 'good' — both checks matter together.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

cluster_features = ["content_id", "impressions_90d", "engagement_rate", "ctr",
                      "avg_position", "days_since_last_update", "word_count"]

lane_slice = df[cluster_features].copy()

print("Shape (rows, columns):", lane_slice.shape)
print("\nOne row = one content page. Here are 5 real rows:")
lane_slice.head(5)

Shape (rows, columns): (30000, 7)

One row = one content page. Here are 5 real rows:


,content_id,impressions_90d,engagement_rate,ctr,avg_position,days_since_last_update,word_count
0,content_304f48230142,3803,5.88,0.76,10.6,20,3221.0
1,content_a1fb4e703a9e,15320,0.00,0.05,20.3,25,2481.0
2,content_9aa793d4d895,12581,0.00,0.09,36.5,20,3515.0
3,content_331d6c4de07b,11751,1.28,0.49,6.2,22,NaN
4,content_d99b7a2d90ca,19140,0.00,0.13,44.0,14,2803.0


In [3]:
print("Once K-Means runs (next assignment), each row will also get a 'cluster' column:")
print("e.g. content_id=1234, cluster=2  ->  meaning: this page belongs to whatever")
print("archetype cluster 2 turns out to represent, based on its feature averages.")
lane_slice.describe()

Once K-Means runs (next assignment), each row will also get a 'cluster' column:
e.g. content_id=1234, cluster=2  ->  meaning: this page belongs to whatever
archetype cluster 2 turns out to represent, based on its feature averages.


,impressions_90d,engagement_rate,ctr,avg_position,days_since_last_update,word_count
count,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,22301.000000
mean,5200.366300,2.534520,0.510733,16.34238,46.098300,3107.760325
std,16838.019547,8.310096,3.279162,15.21679,42.078709,1452.382598
min,1.000000,0.000000,0.000000,0.00000,1.000000,8.000000
25%,81.000000,0.000000,0.000000,6.20000,20.000000,2413.000000
50%,731.000000,0.000000,0.070000,10.80000,20.000000,2877.000000
75%,3615.250000,1.350000,0.290000,22.30000,104.000000,3666.000000
max,517715.000000,100.000000,100.000000,245.00000,373.000000,9546.000000


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (like the starter baseline's stale_visible_page or low_ctr_visible_page reason codes) can only check one or two conditions at a time — for example, "is days_since_last_update >= 180 AND impressions_90d >= 500?" That works for finding one specific pattern, but archetypes are defined by the combination and relative balance of six or more numeric signals at once, not a simple threshold on one or two of them.

For example, two pages could both have high impressions, but one has high engagement and one has low engagement — a single if-statement checking impressions alone would lump them together, even though they represent very different situations (one needs monitoring, the other needs an engagement fix). Writing enough nested if-statements to capture every meaningful combination of six features would essentially mean hand-designing every cluster myself, which defeats the purpose — I'd just be encoding my own assumptions rather than discovering what the data actually shows. K-Means instead looks at the mathematical distance between pages across all features simultaneously, and lets genuinely similar pages group together, including combinations a human might not think to check for.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.